# ResNet X-Ray Classification - Local Dataset

This notebook implements a ResNet-based X-ray classification model using the local dataset in `../dataset/ra/`.

In [1]:
# Install required packages
!pip -q install torch torchvision matplotlib pandas openpyxl pillow scikit-learn seaborn

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import glob
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Set up paths for local dataset
DATA_PATH = os.path.abspath("../dataset/ra")
TRAIN_PATH = os.path.join(DATA_PATH, "Segmentation", "train")
TEST_PATH = os.path.join(DATA_PATH, "Segmentation", "test")
METADATA_PATH = os.path.join(DATA_PATH, "Metadata.xlsx")

print(f"Dataset path: {DATA_PATH}")
print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Metadata path: {METADATA_PATH}")

# Verify paths exist
for path, name in [(DATA_PATH, "Dataset"), (TRAIN_PATH, "Train"), (TEST_PATH, "Test"), (METADATA_PATH, "Metadata")]:
    if os.path.exists(path):
        print(f"✓ {name} path exists")
    else:
        print(f"✗ {name} path missing: {path}")

In [ ]:
# Load metadata
try:
    metadata = pd.read_excel(METADATA_PATH)
    print(f"Metadata shape: {metadata.shape}")
    print("\nMetadata columns:")
    print(metadata.columns.tolist())
    print("\nFirst few rows:")
    print(metadata.head())
except Exception as e:
    print(f"Error loading metadata: {e}")
    metadata = None

In [ ]:
# Explore dataset structure
train_files = glob.glob(os.path.join(TRAIN_PATH, "*.bmp"))
test_files = glob.glob(os.path.join(TEST_PATH, "*.bmp"))

print(f"Training images: {len(train_files)}")
print(f"Test images: {len(test_files)}")

if train_files:
    print(f"\nSample training files:")
    for i, file in enumerate(train_files[:5]):
        print(f"  {i+1}. {os.path.basename(file)}")
        
if test_files:
    print(f"\nSample test files:")
    for i, file in enumerate(test_files[:5]):
        print(f"  {i+1}. {os.path.basename(file)}")

In [ ]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
NUM_CLASSES = 2  # Adjust based on your classification task

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print(f"\nConfiguration:")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Number of epochs: {NUM_EPOCHS}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Data transforms defined:")
print(f"Training transforms: {len(train_transform.transforms)} steps")
print(f"Validation/Test transforms: {len(val_test_transform.transforms)} steps")

In [ ]:
# Custom Dataset class
class XRayDataset(Dataset):
    def __init__(self, image_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        if self.labels is not None:
            label = self.labels[idx]
            return image, label
        else:
            return image

print("Custom XRayDataset class defined successfully!")

In [ ]:
# Create sample datasets (you'll need to modify this based on your labeling strategy)
# For now, creating dummy labels - you should replace this with actual label extraction

# Example: Create labels based on filename patterns or metadata
def extract_labels_from_metadata(image_paths, metadata_df):
    """
    Extract labels from metadata based on image filenames
    Modify this function based on your actual labeling scheme
    """
    labels = []
    for path in image_paths:
        filename = os.path.basename(path)
        # Example: extract patient ID and look up in metadata
        # You'll need to implement this based on your metadata structure
        
        # For demonstration, using a simple pattern
        # Replace this with actual logic
        if 'P00' in filename:  # Example pattern
            labels.append(1)  # RA positive
        else:
            labels.append(0)  # RA negative
    
    return labels

# Extract labels (modify based on your needs)
if metadata is not None:
    train_labels = extract_labels_from_metadata(train_files, metadata)
    test_labels = extract_labels_from_metadata(test_files, metadata)
else:
    # Fallback: create dummy labels
    print("Warning: Using dummy labels. Please implement proper label extraction.")
    train_labels = [i % 2 for i in range(len(train_files))]  # Alternating 0,1
    test_labels = [i % 2 for i in range(len(test_files))]    # Alternating 0,1

print(f"Train labels distribution: {np.bincount(train_labels)}")
print(f"Test labels distribution: {np.bincount(test_labels)}")

In [ ]:
# Split training data into train and validation
train_paths, val_paths, train_labels_split, val_labels_split = train_test_split(
    train_files, train_labels, test_size=0.2, random_state=42, stratify=train_labels
)

print(f"Dataset splits:")
print(f"Training: {len(train_paths)} images")
print(f"Validation: {len(val_paths)} images")
print(f"Test: {len(test_files)} images")

# Create datasets
train_dataset = XRayDataset(train_paths, train_labels_split, train_transform)
val_dataset = XRayDataset(val_paths, val_labels_split, val_test_transform)
test_dataset = XRayDataset(test_files, test_labels, val_test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\nData loaders created:")
print(f"Train loader: {len(train_loader)} batches")
print(f"Validation loader: {len(val_loader)} batches")
print(f"Test loader: {len(test_loader)} batches")

In [ ]:
# Visualize sample images
def visualize_samples(dataloader, num_samples=8):
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    axes = axes.ravel()
    
    # Get a batch of images
    images, labels = next(iter(dataloader))
    
    for i in range(min(num_samples, len(images))):
        # Denormalize the image for visualization
        img = images[i]
        img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        img = torch.clamp(img, 0, 1)
        
        axes[i].imshow(img.permute(1, 2, 0))
        axes[i].set_title(f'Label: {labels[i].item()}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visualizing sample training images:")
visualize_samples(train_loader)

In [ ]:
# Define ResNet model
class ResNetXRayClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(ResNetXRayClassifier, self).__init__()
        
        # Load pre-trained ResNet18
        self.resnet = models.resnet18(pretrained=pretrained)
        
        # Replace the final fully connected layer
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        return self.resnet(x)

# Create model
model = ResNetXRayClassifier(num_classes=NUM_CLASSES, pretrained=True)
model = model.to(device)

print(f"Model created and moved to {device}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print("Training setup:")
print(f"Loss function: {criterion.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate scheduler: {scheduler.__class__.__name__}")

In [ ]:
# Training function
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        # Calculate metrics
        train_loss_avg = train_loss / len(train_loader)
        val_loss_avg = val_loss / len(val_loader)
        train_accuracy = 100 * train_correct / train_total
        val_accuracy = 100 * val_correct / val_total
        
        train_losses.append(train_loss_avg)
        val_losses.append(val_loss_avg)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        
        print(f'Epoch [{epoch+1}/{num_epochs}]:')
        print(f'  Train Loss: {train_loss_avg:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'  Val Loss: {val_loss_avg:.4f}, Val Acc: {val_accuracy:.2f}%')
        print('-' * 60)
        
        scheduler.step()
    
    return train_losses, val_losses, train_accuracies, val_accuracies

print("Training function defined. Ready to start training!")

In [ ]:
# Train the model
print("Starting training...")
print(f"Training on {device} with {len(train_loader)} batches per epoch")
print("=" * 60)

train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler, NUM_EPOCHS, device
)

print("Training completed!")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot losses
ax1.plot(train_losses, label='Training Loss', marker='o')
ax1.plot(val_losses, label='Validation Loss', marker='s')
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Plot accuracies
ax2.plot(train_accuracies, label='Training Accuracy', marker='o')
ax2.plot(val_accuracies, label='Validation Accuracy', marker='s')
ax2.set_title('Training and Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Final Training Accuracy: {train_accuracies[-1]:.2f}%")
print(f"Final Validation Accuracy: {val_accuracies[-1]:.2f}%")

In [ ]:
# Evaluate on test set
def evaluate_model(model, test_loader, device):
    model.eval()
    test_correct = 0
    test_total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    test_accuracy = 100 * test_correct / test_total
    return test_accuracy, all_predictions, all_labels

# Evaluate the model
test_accuracy, predictions, true_labels = evaluate_model(model, test_loader, device)

print(f"Test Accuracy: {test_accuracy:.2f}%")

# Classification report
print("\nClassification Report:")
print(classification_report(true_labels, predictions, target_names=['Class 0', 'Class 1']))

# Confusion Matrix
cm = confusion_matrix(true_labels, predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Save the model
model_save_path = "resnet_xray_classifier.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'train_accuracies': train_accuracies,
    'val_accuracies': val_accuracies,
    'test_accuracy': test_accuracy,
    'config': {
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'num_classes': NUM_CLASSES
    }
}, model_save_path)

print(f"Model saved to {model_save_path}")
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print(f"Final Test Accuracy: {test_accuracy:.2f}%")
print("="*60)